# MobileNetV2 Partial Fine Tuning

Fine tuning for MobileNetV2 model

Note: must have `CNR-EXT-Patches-150x150/` and `splits` in `data-4620-project` directory (see google drive)

In [1]:
from pathlib import Path

import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().resolve()
PATCHES_ROOT = PROJECT_ROOT / 'CNR-EXT-Patches-150x150' / 'PATCHES'
SPLITS_ROOT = PROJECT_ROOT / 'splits' / 'CNRPark-EXT'

if not PATCHES_ROOT.is_dir():
    raise FileNotFoundError(f'Patch directory not found: {PATCHES_ROOT}')
if not SPLITS_ROOT.is_dir():
    raise FileNotFoundError(f'Split directory not found: {SPLITS_ROOT}')

device = 'mps' if torch.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Project root: {PROJECT_ROOT}')
print(f'Using device: {device}')

Project root: /Users/elliepearson/data-4620/data-4620-project
Using device: mps


## Load the official CNR-EXT splits

Each split row contains an image path relative to `PATCHES/` and a binary occupancy label.

In [2]:
BATCH_SIZE = 32

# resize images to 224x224 (compatible with MobileNetV2)
# convert to PIL image to tensor
# normalizes color channels like MobileNetV2
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

# creates dataset based on provided split name
class CNRDataset(Dataset):
    # stores image path in CNRPARK-EXT and label
    def __init__(self, split_name):
        split_file = SPLITS_ROOT / f'{split_name}.txt'
        with split_file.open() as file:
            self.samples = [
                (PATCHES_ROOT / path, float(label)) 
                for path, label in (line.strip().split() for line in file)
            ]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, label = self.samples[index]
        with Image.open(image_path) as image:
            image = image.convert('RGB')
            image = image_transform(image)
        return image, torch.tensor(label, dtype=torch.float32)

# create datasets and dataloaders
train_dataset = CNRDataset('train')
val_dataset = CNRDataset('val')
test_dataset = CNRDataset('test')

train_loader = DataLoader(train_dataset, shuffle=True, batch_size=BATCH_SIZE)
val_loader = DataLoader(val_dataset, shuffle=False, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, shuffle=False, batch_size=BATCH_SIZE)

print(f'{len(train_dataset):,} train | {len(val_dataset):,} validation | {len(test_dataset):,} test images')

94,493 train | 18,647 validation | 31,825 test images


## Fine Tune MobileNetV2 MLP classifier

Train only classifier head to output 0 (free) or 1 (busy). Freeze ModelNetV2 CNN weights (partial fine tuning)

In [ ]:
weights = models.MobileNet_V2_Weights.DEFAULT
model = models.mobilenet_v2(weights=weights)

for parameter in model.features.parameters():
    parameter.requires_grad = False

# train linear layer to make binary output, potentially turn into MLP?
model.classifier[1] = nn.Linear(model.last_channel, 1)
model = model.to(device)

## Train and evaluate


In [ ]:
def evaluate(loader, description):
    model.eval()
    correct = total = 0
    all_labels, all_probabilities = [], []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc=description):
            images = images.to(device)
            labels = labels.to(device)
            probabilities = torch.sigmoid(model(images).squeeze(1))
            predictions = (probabilities >= 0.5).float()

            correct += (predictions == labels).sum().item()
            total += labels.size(0)
            all_labels.extend(labels.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())

    return correct / total

def train(model, criterion, opt, lr=1e-3, epochs=3):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for images, labels in tqdm(train_loader, desc=f'Epoch {epoch + 1}/{epochs}'):
            images = images.to(device)
            labels = labels.to(device)

            opt.zero_grad()
            logits = model(images).squeeze(1)
            loss = criterion(logits, labels)
            loss.backward()
            opt.step()
            running_loss += loss.item()

        val_accuracy = evaluate(val_loader, 'Validation')
        print(f'Epoch {epoch + 1}: loss={running_loss / len(train_loader):.4f}, val accuracy={val_accuracy:.2%}')
    
    return model


In [8]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3)

model = train(model, criterion, optimizer)
test_accuracy = evaluate(test_loader, 'Test')
print(f'Test accuracy: {test_accuracy:.4%}')

Epoch 1/3:   0%|          | 0/2953 [00:00<?, ?it/s]

Validation:   0%|          | 0/583 [00:00<?, ?it/s]

Epoch 1: loss=0.1165, val accuracy=96.89%


Epoch 2/3:   0%|          | 0/2953 [00:00<?, ?it/s]

Validation:   0%|          | 0/583 [00:00<?, ?it/s]

Epoch 2: loss=0.0907, val accuracy=97.60%


Epoch 3/3:   0%|          | 0/2953 [00:00<?, ?it/s]

Validation:   0%|          | 0/583 [00:00<?, ?it/s]

Epoch 3: loss=0.0887, val accuracy=97.47%


Test:   0%|          | 0/995 [00:00<?, ?it/s]

Test accuracy: 98.0801%
